# Detector de residuos – NTP 900.058 (7 colores)
Entrenamiento de **YOLO11n (PyTorch / Ultralytics)** en Google Colab con datasets de Roboflow reagrupados en los 7 colores.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`.

## 1. Instalar librerías y verificar GPU

In [ ]:
!pip install -q "ultralytics>=8.3,<8.4" roboflow
import torch
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Configuración
- Pega tu **API key privada** de Roboflow.
- `version`: número de versión del dataset (pestaña *Dataset* en Roboflow Universe).
- `por_defecto`: si se indica, **todas** las clases de ese dataset van a ese color (útil para el de residuos médicos).
- Descomenta la última línea cuando tengas tu propio dataset de fotos con la webcam.

In [ ]:
ROBOFLOW_API_KEY = "PEGA_AQUI_TU_API_KEY"

DATASETS = [
    # (workspace, proyecto, version, alias, por_defecto)
    ("material-identification", "garbage-classification-3", 2, "gc3", None),
    ("trash-dataset-for-oriented-bounded-box", "trash-detection-1fjjc", 14, "taco", None),
    ("tanim", "medical-waste-detection", 1, "medico", "rojo_peligrosos"),
    # ("TU_WORKSPACE", "residuos-webcam", 1, "propias", None),
]

CLASES = [
    "azul_papel_carton",
    "amarillo_metales",
    "blanco_plasticos",
    "gris_vidrio",
    "marron_organicos",
    "rojo_peligrosos",
    "negro_generales",
]

## 3. Descargar datasets desde Roboflow (formato YOLO)

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

rutas = {}
for ws, proj, ver, alias, _ in DATASETS:
    print(f"\n=== {alias}: {ws}/{proj} v{ver} ===")
    ds = rf.workspace(ws).project(proj).version(ver).download(
        "yolov8", location=f"/content/datasets/{alias}")
    rutas[alias] = ds.location

## 4. Mapeo de clases → 7 colores (NTP 900.058)
Primero se busca el nombre exacto; si no está, se aplican reglas por palabra clave **en orden** (peligrosos primero).
`DESCARTAR` elimina esas cajas. Revisa la tabla que imprime la celda siguiente y ajusta lo que necesites.

In [ ]:
DESCARTAR = "DESCARTAR"

MAPEO_EXACTO = {
    # Nombres propios (tu dataset de webcam) se mantienen
    **{c: c for c in CLASES},
    # GARBAGE CLASSIFICATION 3
    "biodegradable": "marron_organicos",
    "cardboard": "azul_papel_carton",
    "paper": "azul_papel_carton",
    "glass": "gris_vidrio",
    "metal": "amarillo_metales",
    "plastic": "blanco_plasticos",
    # Trash Detection (TACO) - casos especiales
    "waste": DESCARTAR,                 # superclase genérica
    "paper cup": "negro_generales",     # vaso de papel plastificado: no aprovechable
    "crisp packet": "negro_generales",  # envoltura multicapa
    "garbage bag": "negro_generales",
    "carded blister pack": "negro_generales",
    "aluminium blister pack": "negro_generales",
    "squeezable tube": "blanco_plasticos",
    "tupperware": "blanco_plasticos",
    "spread tub": "blanco_plasticos",
    "six pack rings": "blanco_plasticos",
    "toilet tube": "azul_papel_carton",
    "pizza box": "azul_papel_carton",
}

REGLAS = [  # (palabras clave, clase) - el orden importa
    (["battery", "aerosol", "syringe", "needle", "medical", "hazard", "e-waste",
      "ewaste", "cable", "glove", "gloove", "mask", "blood", "lancet", "blade",
      "swab", "infect", "pill", "medicine"], "rojo_peligrosos"),
    (["cigarette", "tissue", "styrofoam", "foam", "shoe", "cloth", "rope",
      "string", "litter", "trash", "general", "rubbish", "blister"], "negro_generales"),
    (["glass"], "gris_vidrio"),
    (["metal", "can", "alumin", "foil", "pop tab", "tin"], "amarillo_metales"),
    (["paper", "cardboard", "carton", "box", "newspaper", "magazine"], "azul_papel_carton"),
    (["plastic", "bottle", "pet", "bag", "cup", "lid", "straw", "container",
      "wrapper", "film", "utensil", "polypropylene"], "blanco_plasticos"),
    (["food", "organic", "bio", "fruit", "vegetable", "peel", "leaf"], "marron_organicos"),
]

def mapear(nombre, por_defecto=None):
    if por_defecto:
        return por_defecto
    n = nombre.strip().lower()
    if n in MAPEO_EXACTO:
        return MAPEO_EXACTO[n]
    for palabras, clase in REGLAS:
        if any(p in n for p in palabras):
            return clase
    return None  # sin mapear -> se descarta (aparece en la tabla)

import yaml
def leer_nombres(raiz):
    with open(f"{raiz}/data.yaml") as f:
        nombres = yaml.safe_load(f)["names"]
    if isinstance(nombres, dict):
        nombres = [nombres[k] for k in sorted(nombres)]
    return nombres

for ws, proj, ver, alias, defecto in DATASETS:
    print(f"\n=== {alias} ===")
    for i, n in enumerate(leer_nombres(rutas[alias])):
        destino = mapear(n, defecto)
        marca = "  <-- SIN MAPEAR" if destino is None else ""
        print(f"{i:>3}  {n:<30} -> {destino}{marca}")

## 5. Unir todo en un solo dataset de 7 clases

In [ ]:
import os, glob, shutil
from collections import Counter

DEST = "/content/residuos7"
shutil.rmtree(DEST, ignore_errors=True)
SPLITS = ["train", "valid", "test"]
for s in SPLITS:
    os.makedirs(f"{DEST}/{s}/images", exist_ok=True)
    os.makedirs(f"{DEST}/{s}/labels", exist_ok=True)

conteo = {s: Counter() for s in SPLITS}
imagenes = Counter()

def a_caja(coords):
    # Si la etiqueta es un polígono (segmentación), lo convierte a caja
    if len(coords) == 4:
        return coords
    xs, ys = coords[0::2], coords[1::2]
    x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
    return [(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]

for ws, proj, ver, alias, defecto in DATASETS:
    raiz = rutas[alias]
    nombres = leer_nombres(raiz)
    remap = {}
    for i, n in enumerate(nombres):
        d = mapear(n, defecto)
        remap[i] = CLASES.index(d) if d in CLASES else None

    for s in SPLITS:
        dir_img = f"{raiz}/{s}/images"
        if not os.path.isdir(dir_img):
            continue
        for ruta_img in glob.glob(f"{dir_img}/*"):
            base, ext = os.path.splitext(os.path.basename(ruta_img))
            ruta_lbl = f"{raiz}/{s}/labels/{base}.txt"
            if not os.path.exists(ruta_lbl):
                continue
            nuevas = []
            for linea in open(ruta_lbl):
                p = linea.split()
                if len(p) < 5:
                    continue
                nuevo = remap.get(int(p[0]))
                if nuevo is None:
                    continue
                caja = a_caja([float(v) for v in p[1:]])
                nuevas.append(f"{nuevo} " + " ".join(f"{v:.6f}" for v in caja))
                conteo[s][CLASES[nuevo]] += 1
            if not nuevas:
                continue  # imagen sin cajas útiles -> se omite
            nombre = f"{alias}_{base}"
            shutil.copy(ruta_img, f"{DEST}/{s}/images/{nombre}{ext}")
            with open(f"{DEST}/{s}/labels/{nombre}.txt", "w") as f:
                f.write("\n".join(nuevas) + "\n")
            imagenes[s] += 1

data = {"path": DEST, "train": "train/images", "val": "valid/images",
        "names": {i: c for i, c in enumerate(CLASES)}}
if imagenes["test"]:
    data["test"] = "test/images"
with open(f"{DEST}/data.yaml", "w") as f:
    yaml.dump(data, f, sort_keys=False)

print("Imágenes:", dict(imagenes))
for s in SPLITS:
    print(f"\n{s}:")
    for c in CLASES:
        print(f"  {c:<20} {conteo[s][c]:>6} cajas")

## 6. Entrenar YOLO11n
~1–2 h en GPU T4. `patience=15` detiene el entrenamiento si deja de mejorar.
Si Colab se desconecta, usa la celda de *reanudar* más abajo.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")  # preentrenado en COCO
model.train(
    data=f"{DEST}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=32,
    patience=15,
    project="/content/runs",
    name="residuos7",
    exist_ok=True,
    plots=True,
)

In [ ]:
# (Solo si se cortó el entrenamiento) Reanudar desde el último checkpoint
# from ultralytics import YOLO
# YOLO("/content/runs/residuos7/weights/last.pt").train(resume=True)

## 7. Evaluar: métricas y matriz de confusión

In [ ]:
best = YOLO("/content/runs/residuos7/weights/best.pt")
split = "test" if imagenes["test"] else "val"
m = best.val(data=f"{DEST}/data.yaml", split=split)
print(f"mAP50: {m.box.map50:.3f} | mAP50-95: {m.box.map:.3f}")
for i, c in enumerate(CLASES):
    try:
        print(f"  {c:<20} AP50 = {m.box.ap50[list(m.box.ap_class_index).index(i)]:.3f}")
    except ValueError:
        print(f"  {c:<20} (sin ejemplos en {split})")

In [ ]:
from IPython.display import Image, display
for f in ["confusion_matrix_normalized.png", "results.png", "PR_curve.png"]:
    ruta = f"/content/runs/residuos7/{f}"
    if os.path.exists(ruta):
        print(f); display(Image(ruta, width=800))

## 8. Probar con una imagen (sube una foto tuya)

In [ ]:
from google.colab import files
subidas = files.upload()
for nombre in subidas:
    r = best.predict(nombre, conf=0.4)[0]
    r.save(filename=f"pred_{nombre}")
    display(Image(f"pred_{nombre}", width=600))

## 9. Descargar el modelo (`best.pt`) y los gráficos

In [ ]:
shutil.copy("/content/runs/residuos7/weights/best.pt", "/content/best.pt")
shutil.make_archive("/content/resultados_residuos7", "zip", "/content/runs/residuos7")
files.download("/content/best.pt")
files.download("/content/resultados_residuos7.zip")